# Занятие 14. Трекинг и поиск по изображениям

За 90 минут: **цветовой трекер → ID и пропуски → BoW/TF-IDF → RANSAC-проверка поиска**.
Три TODO по 10 / 10 / 11 строк. Выберите ядро `iu-intro-cv/.venv`;
зависимости устанавливаются заранее по `docs/setup-guide.md`.
Камера, сеть и готовые файлы данных не нужны: `make.py` создаёт сцену в памяти.

Опора: L14 «Трекинг, поиск и распознавание», 50 слайдов; демо
`lectures/L14-tracking-recognition/demo/02_meanshift_camshift.py`,
`04_iou_association.py`, `07_bow_vocabulary.py`, `08_tfidf_index_verify.py`.
SIFT из L9 и RANSAC из L10 переиспользуем. Поиск подтверждает **экземпляр плоского объекта**;
категорийный детектор здесь не обучаем. Готовая цветовая модель и три небольших
пропуска оставляют время разобраться в причинах ошибок.

План: 15 минут опорного примера, 50 трёх TODO, 15 общего разбора,
10 моста к L15 и вопросов по HW4 (основная клиника по действующему TA-плану).

In [ ]:
import importlib.util
import importlib.metadata
import json
from pathlib import Path
import cv2
import numpy as np
from sklearn.cluster import KMeans
from IPython.display import display, Image

assert cv2.__version__ == '4.14.0', 'Выберите ядро iu-intro-cv/.venv'
assert importlib.metadata.version('opencv-contrib-python') == '4.14.0.94'
cv2.setNumThreads(1)
if 'LAB_DIR' not in globals():
    candidates = (Path.cwd(), Path.cwd()/'labs/lab14-tracking-retrieval')
    LAB_DIR = next(p for p in candidates if (p/'make.py').is_file())
spec = importlib.util.spec_from_file_location('lab14_data', LAB_DIR/'make.py')
data_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_module)
make_scene, array_hashes = data_module.make_scene, data_module.array_hashes


def show(images, labels):
    tiles = []
    for image, label in zip(images, labels):
        view = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR) if image.ndim == 2 else image.copy()
        canvas = np.full((280, 310, 3), 245, np.uint8)
        scale = min(300/view.shape[1], 245/view.shape[0])
        view = cv2.resize(view, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        canvas[28:28+view.shape[0], :view.shape[1]] = view
        cv2.putText(canvas, label, (5, 19), 0, .48, (20, 20, 20), 1, cv2.LINE_AA)
        tiles.append(canvas)
    display(Image(data=cv2.imencode('.png', np.hstack(tiles))[1].tobytes()))


def iou(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    lower, upper = np.maximum(a[:2], b[:2]), np.minimum(a[:2]+a[2:], b[:2]+b[2:])
    inter = float(np.prod(np.maximum(upper-lower, 0)))
    union = float(np.prod(a[2:])+np.prod(b[2:])-inter)
    return inter/union if union > 0 else 0.


def boxes_array(boxes):
    a = np.asarray(boxes, float).reshape(-1, 4)
    if not np.isfinite(a).all() or np.any(a[:, 2:] <= 0):
        raise ValueError('Boxes must be finite xywh with positive size')
    return a


def detect_red(frame):
    mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), (0, 100, 100), (10, 255, 255))
    _, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    return np.array([row[:4] for row in stats[1:] if row[4] > 50], float).reshape(-1, 4)


def color_track(frames, initial, method='CamShift'):
    if method not in ('MeanShift', 'CamShift'):
        raise ValueError('Unknown color tracker')
    x, y, w, h = map(int, initial)
    hsv = cv2.cvtColor(frames[0], cv2.COLOR_BGR2HSV)
    roi = hsv[y:y+h, x:x+w]
    hist = cv2.calcHist([roi], [0], cv2.inRange(roi, (0, 150, 40), (179, 255, 255)), [30], [0, 180])
    cv2.normalize(hist, hist, 0, 255, cv2.NORM_MINMAX)
    window, boxes = (x, y, w, h), []
    term = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, .1)
    for frame in frames:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        bp = cv2.calcBackProject([hsv], [0], hist, [0, 180], 1)
        bp &= cv2.inRange(hsv, (0, 150, 40), (179, 255, 255))
        if method == 'MeanShift':
            _, window = cv2.meanShift(bp, window, term)
            box = window
        else:
            rotated, window = cv2.CamShift(bp, window, term)
            box = cv2.boundingRect(cv2.boxPoints(rotated))
        boxes.append(box)
    return np.array(boxes, float)

## 0–15. Один цвет, два разных вопроса

MeanShift сдвигает окно к центру массы обратной проекции (backprojection).
CamShift также меняет размер окна по моментам. Гистограмма обучается только
на начальной области; GT в трекер не передаётся. В готовой сцене цель увеличивается.
Сравним средний IoU по **всем 40 кадрам**, включая начальный; затем посмотрим
другой ролик, где одинаковые по цвету объекты пересекаются и один исчезает на два кадра.
Цвет помогает найти область, но не доказывает идентичность.

In [ ]:
scene = make_scene()
assert scene['train_desc'].shape[1] == 128
color_results = {name: color_track(scene['color_frames'], scene['color_gt'][0], name)
                 for name in ('MeanShift', 'CamShift')}
color_metrics = {name: float(np.mean([iou(a, b) for a, b in zip(boxes, scene['color_gt'])]))
                 for name, boxes in color_results.items()}
print(json.dumps(color_metrics, indent=2))
assert color_metrics['CamShift'] > color_metrics['MeanShift'] + .2
color_view = scene['color_frames'][-1].copy()
for name, color in [('MeanShift', (240, 100, 30)), ('CamShift', (200, 30, 200))]:
    x, y, w, h = np.rint(color_results[name][-1]).astype(int)
    cv2.rectangle(color_view, (x, y), (x+w-1, y+h-1), color, 2)
    cv2.putText(color_view, name, (x, y-4), 0, .4, color, 1, cv2.LINE_AA)
show([scene['color_frames'][0], color_view, scene['frames'][20]],
     ['Initial color target', 'Tracker windows after growth', 'Two targets overlap'])

## TODO 1 · 15–32. Ассоциация рамок по IoU

Рамка имеет формат `(x,y,w,h)`. `iou` дана выше. Составьте все пары
«прогноз трека — детекция», сортируйте по убыванию IoU (при равенстве по индексам).
Выберите пары с IoU ≥ threshold, используя каждый индекс не более одного раза.
Верните список `(track_index, detection_index)`. Пустой вход даёт пустой список.

В готовом каркасе прогноз постоянной скорости — просто перенос `(x,y)` по двум
последним измерениям с учётом числа прошедших кадров. Пропуск увеличивает `missed`;
после `max_missed=7` трек удаляется. Слитая компонента больше 900 px² считается
неоднозначной и не корректирует историю. Это ограничение **данных целей постоянного
размера**, на произвольном видео такой порог не годится. Жадный алгоритм локален;
при одинаковом внешнем виде и долгой окклюзии он может поменять ID.

In [ ]:
def greedy_match(predicted, detected, threshold=.2):
    """Descending IoU, deterministic ties, one track and detection per pair."""
    predicted, detected = boxes_array(predicted), boxes_array(detected)
    if not 0 < threshold <= 1:
        raise ValueError('IoU threshold must be in (0,1]')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10 строк: IoU всех пар, сортировка, два множества занятых индексов
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────

In [ ]:
def track_boxes(frames, predict=True, max_missed=7):
    """Only observations enter the tracker. No GT or physical IDs are inputs."""
    if max_missed < 0:
        raise ValueError('max_missed must be nonnegative')
    tracks, history, next_id, rejected_merges = [], [], 0, 0
    for t, frame in enumerate(frames):
        detected = detect_red(frame)
        # A fixed size gate is appropriate ONLY for these constant-size targets.
        # A merged component is ambiguous: keep a forecast, do not update appearance/velocity.
        keep = np.prod(detected[:, 2:], axis=1) <= 900
        rejected_merges += int(np.count_nonzero(~keep))
        detected = detected[keep]
        proposals = np.array([tr['box'] + np.r_[tr['velocity']*(t-tr['time']) if predict else [0., 0.], 0., 0.]
                              for tr in tracks]).reshape(-1, 4)
        pairs = greedy_match(proposals, detected)
        used_t, used_d = {i for i, j in pairs}, {j for i, j in pairs}
        for i, j in pairs:
            tr, box = tracks[i], detected[j]
            tr['velocity'] = (box[:2]-tr['box'][:2])/(t-tr['time'])
            tr.update(box=box.copy(), time=t, missed=0)
        for i, tr in enumerate(tracks):
            if i not in used_t:
                tr['missed'] += 1
        tracks = [tr for tr in tracks if tr['missed'] <= max_missed]
        for j, box in enumerate(detected):
            if j not in used_d:
                tracks.append(dict(id=next_id, box=box.copy(), velocity=np.zeros(2), time=t, missed=0))
                next_id += 1
        history.append([dict(id=tr['id'], box=(tr['box'] + np.r_[tr['velocity']*(t-tr['time']) if predict else [0., 0.], 0., 0.]).copy(),
                             observed=tr['time'] == t, missed=tr['missed']) for tr in tracks])
    return dict(history=history, created_ids=next_id, rejected_merges=rejected_merges)


def identity_report(result, truth, visible):
    """One-to-one evaluation, only matched observations, plus full visible denominator."""
    last, switches, matched, eligible = {}, 0, 0, int(np.count_nonzero(visible))
    errors = []
    for rows, gt, visibility in zip(result['history'], truth, visible):
        rows = [r for r in rows if r['observed']]
        valid = np.flatnonzero(visibility)
        observed = np.array([r['box'] for r in rows]).reshape(-1, 4)
        for i, j in greedy_match(gt[valid], observed, .5):
            physical, ident = int(valid[i]), rows[j]['id']
            switches += int(physical in last and last[physical] != ident)
            last[physical] = ident
            matched += 1
            errors.append(np.linalg.norm(gt[physical, :2]+gt[physical, 2:]/2
                                         - observed[j, :2]-observed[j, 2:]/2))
    return dict(id_switches=switches, matched=matched, eligible=eligible,
                coverage=matched/eligible if eligible else 0.,
                center_error_px=float(np.mean(errors)) if errors else None,
                created_ids=result['created_ids'], rejected_merges=result['rejected_merges'])


def tracking_view(frame, rows):
    image = frame.copy()
    for row in rows:
        x, y, w, h = np.rint(row['box']).astype(int)
        color = (50, 220, 50) if row['observed'] else (0, 180, 250)
        cv2.rectangle(image, (x, y), (x+w, y+h), color, 2)
        text = str(row['id']) + (' measured' if row['observed'] else ' forecast')
        cv2.putText(image, text, (x, max(y-4, 10)), 0, .4, color, 1, cv2.LINE_AA)
    return image

In [ ]:
assert greedy_match([[0, 0, 30, 25]], [[5, 0, 30, 25]]) == [(0, 0)]
assert greedy_match([], [[5, 0, 30, 25]]) == []
tracked = track_boxes(scene['frames'])
tracking_metrics = identity_report(tracked, scene['boxes_gt'], scene['visible'])
baseline = identity_report(track_boxes(scene['frames'], predict=False), scene['boxes_gt'], scene['visible'])
print(json.dumps(dict(prediction=tracking_metrics, no_prediction=baseline), indent=2))
assert tracking_metrics['created_ids'] == 2 and tracking_metrics['id_switches'] == 0
assert tracking_metrics['coverage'] > .8
show([tracking_view(scene['frames'][t], tracked['history'][t]) for t in (8, 20, 25)],
     ['Short missing observation', 'Merge: forecasts only', 'Separate observations again'])

## TODO 2 · 32–49. Визуальный словарь и TF-IDF

Готовые локальные дескрипторы SIFT получены из процедурных плакатов.
`train_desc` — **шесть отдельных обучающих изображений**, не запрос и не база.
Обучите k-means с `k=200, n_init=3, max_iter=80, random_state=14`.
Для каждого документа посчитайте слова через готовую `word_counts`.
`TF = counts / max(sum(counts),1)`, `DF = count_nonzero(counts, axis=0)`;
`IDF = log((N+1)/(DF+1))+1`. Умножьте TF на IDF и L2-нормируйте строки.
Нулевая строка остаётся нулевой. IDF вычисляется **только по базе**;
запрос использует уже сохранённый IDF и не меняет модель.

В базе 0–5 — разные плакаты, 6 — переставленные фрагменты плаката 0.
BoW не хранит расположение деталей; проверьте, какой кандидат оказался выше.

In [ ]:
def word_counts(desc, model):
    if len(desc) == 0:
        return np.zeros(model.n_clusters, int)
    return np.bincount(model.predict(np.asarray(desc, np.float32)), minlength=model.n_clusters)


def normalize_rows(values):
    values = np.asarray(values, float)
    return values/np.maximum(np.linalg.norm(values, axis=-1, keepdims=True), 1e-12)


def build_bow(train_desc, database_desc, k=200):
    """Train vocabulary on a separate corpus; IDF uses database only."""
    if k < 2 or len(train_desc) < k or not len(database_desc):
        raise ValueError('Need training descriptors, database and 2 <= k <= len(train_desc)')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10 строк: k-means, частоты слов, DF/IDF и нормировка строк
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


def query_vector(desc, bow):
    counts = word_counts(desc, bow['model'])
    tf = counts/max(int(counts.sum()), 1)
    return normalize_rows(tf*bow['idf'])

In [ ]:
database_desc = [scene[f'db{i}_desc'] for i in range(len(scene['database']))]
database_xy = [scene[f'db{i}_xy'] for i in range(len(scene['database']))]
database_shapes = [im.shape for im in scene['database']]
bow = build_bow(scene['train_desc'], database_desc)
assert bow['counts'].shape == (7, 200)
np.testing.assert_array_equal(bow['counts'].sum(axis=1), [len(d) for d in database_desc])
np.testing.assert_allclose(np.linalg.norm(bow['vectors'], axis=1), 1., atol=1e-12)
q = query_vector(scene['query_desc'], bow)
raw_ranking = np.argsort(-(bow['vectors'] @ q), kind='stable')
print('BoW ranking:', raw_ranking.tolist())
show([scene['query'], scene['database'][raw_ranking[0]], scene['database'][0]],
     ['Query', 'Top BoW candidate', 'Correct planar instance'])

## TODO 3 · 49–65. Геометрическое подтверждение

Сопоставьте дескрипторы L2 через `BFMatcher.knnMatch(k=2)`, примените ratio test 0.75.
Для ≥4 пар вызовите `cv2.findHomography(src,dst,cv2.RANSAC,3.,maxIters=2000,confidence=.995)`.
Направление H: **query → candidate**. При недостатке пар или отсутствии модели — отказ.
Верните результат готовой `geometry_report`, передав маску инлаеров как bool.

Готовая проверка требует ≥12 инлаеров, долю ≥0.35, покрытие выпуклой оболочкой
≥0.12 **в обоих изображениях**, медиану репроекции <3 px и выпуклую проекцию углов
с площадью в диапазоне 0.15…3 площадей кандидата. Это учебные пороги для плоских
плакатов. Один неизменённый фрагмент не должен подтверждать весь объект.
Проверяем топ-5; затем сортируем принятые результаты по числу инлаеров.
Повторим поиск с посторонним текстурным запросом и однотонным запросом без признаков.

In [ ]:
def empty_geometry(matches=0):
    return dict(matches=matches, inliers=0, fraction=0., coverage=0., candidate_coverage=0.,
                reprojection_px=None, accepted=False, H=None)


def geometry_report(src, dst, H, inside, query_shape, candidate_shape):
    if not np.isfinite(H).all() or abs(np.linalg.det(H)) < 1e-12:
        return empty_geometry(len(src))
    n = int(inside.sum())
    if n < 4:
        return empty_geometry(len(src))
    projected = cv2.perspectiveTransform(src[:, None], H)[:, 0]
    error = float(np.median(np.linalg.norm(projected[inside]-dst[inside], axis=1)))
    coverage = float(cv2.contourArea(cv2.convexHull(src[inside]))/np.prod(query_shape[:2]))
    candidate_coverage = float(cv2.contourArea(cv2.convexHull(dst[inside]))/np.prod(candidate_shape[:2]))
    height, width = query_shape[:2]
    corners = np.float32([[0, 0], [width-1, 0], [width-1, height-1], [0, height-1]])
    quad = cv2.perspectiveTransform(corners[:, None], H)
    area = float(abs(cv2.contourArea(quad))/np.prod(candidate_shape[:2]))
    sane = np.isfinite(quad).all() and cv2.isContourConvex(quad) and .15 < area < 3
    accepted = bool(n >= 12 and n/len(src) >= .35 and min(coverage, candidate_coverage) >= .12
                    and error < 3 and sane)
    return dict(matches=len(src), inliers=n, fraction=n/len(src), coverage=coverage,
                candidate_coverage=candidate_coverage, reprojection_px=error, accepted=accepted, H=H)


def verify_pair(query_xy, query_desc, candidate_xy, candidate_desc, query_shape, candidate_shape):
    if len(query_xy) != len(query_desc) or len(candidate_xy) != len(candidate_desc):
        raise ValueError('Point and descriptor lengths differ')
    if min(len(query_desc), len(candidate_desc)) < 4:
        return empty_geometry()
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 11 строк: ratio test и RANSAC-гомография query → candidate
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


def search(desc, xy, shape, database_desc, database_xy, database_shapes, bow, top_k=5):
    if not 1 <= top_k <= len(database_desc):
        raise ValueError('top_k must be between 1 and database size')
    q = query_vector(desc, bow)
    scores = bow['vectors'] @ q
    ranking = np.argsort(-scores, kind='stable')[:top_k]
    checked = []
    for ident in ranking:
        i = int(ident)
        report = verify_pair(xy, desc, database_xy[i], database_desc[i], shape, database_shapes[i])
        checked.append(dict(id=i, score=float(scores[i]), **report))
    accepted = sorted([r for r in checked if r['accepted']], key=lambda r: (-r['inliers'], r['reprojection_px'], r['id']))
    return dict(ranking=ranking.tolist(), scores=scores.tolist(), checked=checked,
                accepted_ids=[r['id'] for r in accepted])


def serial_report(report):
    return {key: value for key, value in report.items() if key != 'H'}


def verification_view(query, candidate, H):
    left, right = cv2.cvtColor(query, cv2.COLOR_GRAY2BGR), cv2.cvtColor(candidate, cv2.COLOR_GRAY2BGR)
    if H is not None:
        h, w = query.shape
        quad = cv2.perspectiveTransform(np.float32([[[0, 0]], [[w-1, 0]], [[w-1, h-1]], [[0, h-1]]]), H)
        if np.isfinite(quad).all() and np.max(np.abs(quad)) < 1e5:
            cv2.polylines(right, [np.rint(quad).astype(np.int32)], True, (0, 180, 0), 2)
    canvas = np.full((max(len(left), len(right)), left.shape[1]+right.shape[1], 3), 245, np.uint8)
    canvas[:len(left), :left.shape[1]] = left
    canvas[:len(right), left.shape[1]:] = right
    if H is not None:
        # Ready L9 matching, drawn only where the supplied H explains the pair.
        xy1, d1 = data_module.features(query)
        xy2, d2 = data_module.features(candidate)
        good = [a for a, b in cv2.BFMatcher(cv2.NORM_L2).knnMatch(d1, d2, k=2)
                if a.distance < .75*b.distance]
        src = np.float32([xy1[m.queryIdx] for m in good]).reshape(-1, 1, 2)
        dst = np.float32([xy2[m.trainIdx] for m in good]).reshape(-1, 2)
        if len(src):
            error = np.linalg.norm(cv2.perspectiveTransform(src, H)[:, 0]-dst, axis=1)
            kept = np.flatnonzero(error < 3.)
            chosen = kept[np.linspace(0, len(kept)-1, min(18, len(kept))).astype(int)] if len(kept) else []
            for j in chosen:
                cv2.line(canvas, tuple(np.rint(src[j, 0]).astype(int)),
                         tuple(np.rint(dst[j]+[query.shape[1], 0]).astype(int)), (30, 160, 30), 1, cv2.LINE_AA)
    return canvas

In [ ]:
retrieval = {}
for name in ('query', 'foreign', 'blank'):
    result = search(scene[name+'_desc'], scene[name+'_xy'], scene[name].shape,
                    database_desc, database_xy, database_shapes, bow)
    retrieval[name] = {**result, 'checked': [serial_report(r) for r in result['checked']]}
assert retrieval['query']['accepted_ids'] == [0]
assert not retrieval['foreign']['accepted_ids'] and not retrieval['blank']['accepted_ids']
assert 6 in retrieval['query']['ranking']
for row in retrieval['query']['checked']:
    print(json.dumps(row))
print('Foreign:', retrieval['foreign']['accepted_ids'], 'Blank:', retrieval['blank']['accepted_ids'])
positive = verify_pair(scene['query_xy'], scene['query_desc'], database_xy[0], database_desc[0],
                       scene['query'].shape, database_shapes[0])
scrambled = verify_pair(scene['query_xy'], scene['query_desc'], database_xy[6], database_desc[6],
                        scene['query'].shape, database_shapes[6])
show([verification_view(scene['query'], scene['database'][0], positive['H']),
      verification_view(scene['query'], scene['database'][6], scrambled['H'])],
     ['Accepted: support across the image', 'Rejected: agreement on a small patch'])
metrics = dict(color=color_metrics, tracking=tracking_metrics, no_prediction=baseline,
               training_descriptors=len(scene['train_desc']), retrieval=retrieval)
print(json.dumps(metrics, indent=2))

## 65–80. Разбор результата и ⭐ обратный индекс

Покажите TA три результата: восстановление ID после пропуска вместе с покрытием,
рейтинг BoW до геометрии, принятый список после геометрии и два отказа.
Зелёная рамка — измерение, жёлтая — прогноз; при пересечении не выдавайте прогноз
за наблюдение. Нулевые ID switches на этой сцене не гарантируют идентичность на другом ролике.

⭐ Реализуйте обратный индекс (inverted file index): для каждого слова храните
только `(document_id, weight)` с ненулевым весом. При запросе обходите postings
только его ненулевых слов и накапливайте `q[word]*weight`.
Проверьте scores и **весь** стабильный рейтинг против `vectors @ q` до `1e-12`:
обычный запрос, нулевой запрос, пустой posting, равные scores, новый seed.
На отдельной разреженной синтетике считайте посещённые postings вместо замера времени.
Индекс ускоряет вычисление того же сходства; геометрическая проверка остаётся обязательной.

## 80–90. Мост к L15 и основная клиника HW4

В L15 проверенные соответствия становятся входом в E/F, триангуляцию и PnP.
H подходит плоской сцене; для произвольной 3D-сцены одной гомографии недостаточно.
ID рамки — не ID 3D-точки: для реконструкции нужны устойчивые **локальные** соответствия
и видимость. TF-IDF выбирает кандидатов, но не даёт метрического масштаба.

По текущему TA-плану здесь основная клиника HW4 и вопросы HW3/HW4.
Первые 3 минуты: связать поиск и геометрию с L15. Затем 7 минут: открыть публичные
`homeworks/hw4-calib3d/README.md`, `sample_data/README.md`, `case1/expected.json`
и свой ноутбук lab11/lab12. Обсудить худший калибровочный кадр, RMS в px,
disparity `/16`, базу в метрах, GT-маску и пропуски; не подменять глубину цветом/IoU.
Вопросы HW3: нормировка H и ошибка репроекции. Новых домашних заданий, сроков
и условий оценивания эта лабораторная не вводит. При нехватке времени общие вопросы
собираются к уже предусмотренному разбору с лектором.

Для самостоятельной проверки: чем отличаются детекция и трекинг? Почему долгий
пропуск разрушает ID? Какая информация теряется в BoW? Почему четыре инлаера в
одном углу недостаточны? Почему верный H ещё не даёт глубину?
Опоры — L14 и демо 02/04/07/08; L9 (SIFT), L10 (RANSAC/H), lab11/lab12 (HW4).